# Notebook 4.5 - EfficientNet-B0 Baseline

Notebook này chạy riêng model `efficientnet_b0` với 5 seed `42-46` để lấy số liệu bảng so sánh.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent

sys.path.append(str(repo_root))

from src.baseline_experiment import run_baseline_suite
from src.baseline_protocol import (
    DEFAULT_BASELINE_SEEDS,
    build_output_dir,
    ensure_seed_completeness,
    filter_runs_for_run,
    make_fair_train_config,
)

data_dir = repo_root / 'data' / 'features' / 'mel'
base_output_dir = repo_root / 'data' / 'models' / 'baselines'

RUN_ID = 'paper_v1'
output_dir = build_output_dir(base_output_dir=base_output_dir, run_id=RUN_ID)

print(f'Repo root: {repo_root}')
print(f'Data dir: {data_dir}')
print(f'Output dir: {output_dir}')
print(f'Run ID: {RUN_ID}')


## Protocol

- Model: `efficientnet_b0`
- Seeds: `42, 43, 44, 45, 46`
- Mục tiêu: lấy số liệu cho bảng, không vẽ biểu đồ.


In [ ]:
selected_model = 'efficientnet_b0'
model_label = 'EfficientNet-B0'
selected_seeds = DEFAULT_BASELINE_SEEDS
USE_CACHED_RESULTS = False  # True: chỉ đọc CSV trong đúng run_id, không train

config = make_fair_train_config(
    data_dir=data_dir,
    base_output_dir=base_output_dir,
    run_id=RUN_ID,
    model_names=(selected_model,),
    seeds=selected_seeds,
    include_existing_runs=True,
    skip_completed_runs=True,
)

config


In [ ]:
def build_single_summary(runs: pd.DataFrame) -> pd.DataFrame:
    if runs.empty:
        return pd.DataFrame()
    return (
        runs.groupby('model', as_index=False)
        .agg(
            n_runs=('seed', 'nunique'),
            params=('params', 'mean'),
            test_accuracy_mean=('test_accuracy', 'mean'),
            test_accuracy_std=('test_accuracy', 'std'),
            macro_f1_mean=('macro_f1', 'mean'),
            macro_f1_std=('macro_f1', 'std'),
            weighted_f1_mean=('weighted_f1', 'mean'),
            weighted_f1_std=('weighted_f1', 'std'),
            train_seconds_mean=('train_seconds', 'mean'),
            infer_seconds_mean=('infer_seconds', 'mean'),
        )
        .fillna(0.0)
    )

if USE_CACHED_RESULTS:
    print('[MODE] Cached: đọc CSV hiện có trong run_id, không train lại.')
    runs_df = pd.read_csv(output_dir / 'baseline_runs.csv')
    history_store = {}
    metadata = {
        'mode': 'cached',
        'run_id': RUN_ID,
        'runs_csv': str(output_dir / 'baseline_runs.csv'),
    }
else:
    print('[MODE] Train: train phần thiếu trong run_id hiện tại.')
    runs_df, _, history_store, metadata = run_baseline_suite(config)

runs_df = filter_runs_for_run(
    runs_df=runs_df,
    model_names=(selected_model,),
    run_id=RUN_ID,
)
ensure_seed_completeness(
    runs_df=runs_df,
    model_names=(selected_model,),
    seeds=selected_seeds,
)
summary_df = build_single_summary(runs_df)

runs_selected_path = output_dir / f'baseline_runs_{selected_model}.csv'
summary_selected_path = output_dir / f'baseline_summary_{selected_model}.csv'

runs_df.to_csv(runs_selected_path, index=False)
summary_df.to_csv(summary_selected_path, index=False)

print('Metadata:')
print(metadata)
print('Saved:', runs_selected_path)
print('Saved:', summary_selected_path)


In [ ]:
display_cols = [
    'model', 'seed', 'params', 'best_epoch', 'test_accuracy',
    'macro_f1', 'weighted_f1', 'infer_seconds', 'train_seconds'
]

runs_df[display_cols].sort_values('seed')


In [ ]:
params_m = runs_df['params'].mean() / 1_000_000

acc_mean = runs_df['test_accuracy'].mean()
acc_std = runs_df['test_accuracy'].std(ddof=1)
macro_f1_mean = runs_df['macro_f1'].mean()
macro_f1_std = runs_df['macro_f1'].std(ddof=1)
weighted_f1_mean = runs_df['weighted_f1'].mean()
weighted_f1_std = runs_df['weighted_f1'].std(ddof=1)
infer_mean = runs_df['infer_seconds'].mean()
infer_std = runs_df['infer_seconds'].std(ddof=1)
train_mean = runs_df['train_seconds'].mean()
train_std = runs_df['train_seconds'].std(ddof=1)

latex_row = (
    f"{model_label} & {params_m:.2f} & "
    f"{acc_mean:.2f}$\\pm${acc_std:.2f} & "
    f"{macro_f1_mean:.4f}$\\pm${macro_f1_std:.4f} & "
    f"{weighted_f1_mean:.4f}$\\pm${weighted_f1_std:.4f} & "
    f"{infer_mean:.4f}$\\pm${infer_std:.4f} & "
    f"{train_mean:.1f}$\\pm${train_std:.1f} \\\\"
)

result_df = pd.DataFrame([
    {
        'model_label': model_label,
        'params_m': params_m,
        'test_accuracy_mean': acc_mean,
        'test_accuracy_std': acc_std,
        'macro_f1_mean': macro_f1_mean,
        'macro_f1_std': macro_f1_std,
        'weighted_f1_mean': weighted_f1_mean,
        'weighted_f1_std': weighted_f1_std,
        'infer_seconds_mean': infer_mean,
        'infer_seconds_std': infer_std,
        'train_seconds_mean': train_mean,
        'train_seconds_std': train_std,
    }
])

metrics_path = output_dir / f'baseline_table_metrics_{selected_model}.csv'
result_df.to_csv(metrics_path, index=False)

print(latex_row)
print('Saved:', metrics_path)
result_df
